# 01 - Exploratory Data Analysis
**Project:** Financial Complaint Risk Intelligence System
**Author:** Sanjeev Kumar | IIT Bombay
**Dataset:** CFPB Consumer Complaint Database

In [ ]:
import matplotlib
matplotlib.use('Agg')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns, warnings, re, os
warnings.filterwarnings('ignore')
os.makedirs('../reports', exist_ok=True)
os.makedirs('../data', exist_ok=True)

local_path = '../data/complaints_200k.csv'
if os.path.exists(local_path):
    print("Loading from local cache...")
    df = pd.read_csv(local_path, low_memory=False)
else:
    import requests
    url = "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"
    zip_path = '../data/complaints.csv.zip'
    print("Downloading CFPB data (streaming)...")
    with requests.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(zip_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    print("Reading 200 k rows from ZIP...")
    df = pd.read_csv(zip_path, compression='zip', low_memory=False, nrows=200000)
    df.to_csv(local_path, index=False)
    print("Cached locally.")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

In [ ]:
print(f"Date range: {df['Date received'].min()} to {df['Date received'].max()}")
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
print("PRODUCT DISTRIBUTION:")
print(df['Product'].value_counts())

In [ ]:
has_narrative = df['Consumer complaint narrative'].notna()
print(f"With narrative:    {has_narrative.sum():,} ({has_narrative.mean():.1%})")
print(f"Without narrative: {(~has_narrative).sum():,} ({(~has_narrative).mean():.1%})")

In [ ]:
print("TIMELY RESPONSE:")
print(df['Timely response?'].value_counts())
print("\nCOMPANY RESPONSE:")
print(df['Company response to consumer'].value_counts())
is_untimely = df['Timely response?'] == 'No'
is_monetary = df['Company response to consumer'].str.strip() == 'Closed with monetary relief'
print(f"\nUntirely: {is_untimely.sum():,} ({is_untimely.mean():.2%})")
print(f"Monetary relief (exact): {is_monetary.sum():,} ({is_monetary.mean():.2%})")

In [ ]:
df_text = df[df['Consumer complaint narrative'].notna()].copy()
def clean_text(text):
    if pd.isna(text): return ""
    text = str(text).lower()
    text = re.sub(r'x{2,}', 'XXXX', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
df_text['narrative_clean'] = df_text['Consumer complaint narrative'].apply(clean_text)
df_text['text_length'] = df_text['narrative_clean'].str.split().str.len()
print("TEXT LENGTH:")
print(df_text['text_length'].describe())
print(f"Complaints > 256 tokens: {(df_text['text_length']>200).mean():.1%}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
product_counts = df_text['Product'].value_counts()[:8]
axes[0][0].barh(range(len(product_counts)), product_counts.values, color='steelblue')
axes[0][0].set_yticks(range(len(product_counts)))
axes[0][0].set_yticklabels([p[:35] for p in product_counts.index], fontsize=8)
axes[0][0].set_title('Top Products', fontweight='bold')
axes[0][1].hist(df_text['text_length'].clip(0, 600), bins=50, color='steelblue', alpha=0.7)
axes[0][1].axvline(x=256, color='red', linestyle='--', label='256 tokens')
axes[0][1].set_title('Text Length Distribution', fontweight='bold')
axes[0][1].legend()
resp = df['Company response to consumer'].value_counts()[:5]
axes[1][0].bar(range(len(resp)), resp.values, color='coral')
axes[1][0].set_xticks(range(len(resp)))
axes[1][0].set_xticklabels([r[:20] for r in resp.index], rotation=45, ha='right', fontsize=8)
axes[1][0].set_title('Company Responses', fontweight='bold')
df['Date received'] = pd.to_datetime(df['Date received'])
monthly = df.groupby(df['Date received'].dt.to_period('Y')).size()
axes[1][1].plot(monthly.index.astype(str), monthly.values, marker='o', color='steelblue')
axes[1][1].set_title('Annual Complaint Volume', fontweight='bold')
axes[1][1].tick_params(axis='x', rotation=45)
plt.suptitle('CFPB EDA', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/01_eda.png', dpi=150, bbox_inches='tight')
plt.close()
print("EDA complete!")

## Summary

| Metric | Value |
|--------|-------|
| Total rows | 200,000 |
| With narrative | ~48,815 (24.4%) |
| Risk label | untimely response = 1.07% |
| Mean text length | 177 words |

**Next → 02_baseline_models.ipynb**